In [16]:
!pip install pandas numpy matplotlib seaborn

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [19]:
#/content/olist_geolocation_dataset.csv
#/content/olist_order_items_dataset.csv
#/content/olist_order_payments_dataset.csv
#/content/olist_order_reviews_dataset.csv
#/content/olist_orders_dataset.csv
#/content/olist_products_dataset.csv
#/content/olist_sellers_dataset.csv
#/content/product_category_name_translation.csv

Load Data into Pandas

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load CSVs
orders = pd.read_csv('/content/olist_orders_dataset.csv', engine='python')
order_items = pd.read_csv('/content/olist_order_items_dataset.csv', engine='python')
customers = pd.read_csv('/content/olist_customers_dataset.csv', engine='python')
products = pd.read_csv('/content/olist_products_dataset.csv', engine='python')  # Optional for categories
product_category_name=pd.read_csv('/content/product_category_name_translation.csv', engine='python')
sellers = pd.read_csv('/content/olist_sellers_dataset.csv', engine='python')
# Preview
print(orders.head())
print(orders.shape)  # Should show ~99K rows, 8 columns

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08

Extract: Merge Datasets:

In [21]:
# Merge orders with order_items on 'order_id'
merged_df = pd.merge(orders, order_items, on='order_id', how='left')
# Merge with customers on 'customer_id'
merged_df = pd.merge(merged_df, customers, on='customer_id', how='left')
# Optional: Merge with products for categories
merged_df = pd.merge(merged_df, products[['product_id', 'product_category_name']], on='product_id', how='left')

print(merged_df.shape)
print(merged_df.head())

(113425, 19)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00     

In [22]:
print("Orders shape:", orders.shape)  # Should be ~99,441 rows, 8 columns
print("Order items shape:", order_items.shape)  # Should be ~112,650 rows, 7 columns
print("Customers shape:", customers.shape)  # Should be ~99,441 rows, 5 columns
print("Products shape:", products.shape)  # Should be ~32,951 rows, 9 columns (optional)

Orders shape: (99441, 8)
Order items shape: (112650, 7)
Customers shape: (99441, 5)
Products shape: (32951, 9)


In [23]:
print("Unique order_ids in orders:", orders['order_id'].nunique())  # ~99K
print("Unique order_ids in order_items:", order_items['order_id'].nunique())  # Should be ~83K or less
print("Nulls in orders['order_id']:", orders['order_id'].isnull().sum())  # Should be 0
print("Nulls in order_items['order_id']:", order_items['order_id'].isnull().sum())  # Should be 0

Unique order_ids in orders: 99441
Unique order_ids in order_items: 98666
Nulls in orders['order_id']: 0
Nulls in order_items['order_id']: 0


Transform and Clean Data:

1.Handle missing values
2.Convert dates
3.Create revenue column
4.Remove duplicates and filter
5.Check data types

In [24]:
print("Before cleaning:", merged_df.shape)
merged_df.dropna(inplace=True)  # Drops rows with nulls
print("After dropna:", merged_df.shape)
merged_df['order_purchase_timestamp'] = pd.to_datetime(merged_df['order_purchase_timestamp'], errors='coerce')
merged_df['revenue'] = merged_df['price'] + merged_df['freight_value']
merged_df.drop_duplicates(inplace=True)
merged_df = merged_df[merged_df['order_status'] == 'delivered']
print("After filtering and cleaning:", merged_df.shape)

Before cleaning: (113425, 19)
After dropna: (108644, 19)
After filtering and cleaning: (108637, 20)


In [25]:
print(merged_df.dtypes)  # Ensure 'revenue' is float, dates are datetime

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                        object
order_delivered_carrier_date             object
order_delivered_customer_date            object
order_estimated_delivery_date            object
order_item_id                           float64
product_id                               object
seller_id                                object
shipping_limit_date                      object
price                                   float64
freight_value                           float64
customer_unique_id                       object
customer_zip_code_prefix                  int64
customer_city                            object
customer_state                           object
product_category_name                    object
revenue                                 float64
dtype: object


Load: Export Cleaned Data

In [26]:
merged_df.to_csv('/content/cleaned_ecommerce_data.csv', index=False)

In [27]:
# After loading product_category_name
products = pd.merge(products, product_category_name, on='product_category_name', how='left')
# Then re-merge with merged_df if desired
merged_df = pd.merge(merged_df, products[['product_id', 'product_category_name_english']], on='product_id', how='left')

In [29]:
print("After filtering and cleaning:", merged_df.shape)

After filtering and cleaning: (108615, 21)


In [30]:
merged_df.to_csv('/content/cleaned_ecommerce_data1.csv', index=False)